<a href="https://colab.research.google.com/github/Komil-jon/ev-battery-vision-pipeline/blob/main/notebooks/colab_train_detector_curated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train the EV detector on the CURATED dataset (GPU)

Curated after the annotation-consistency audit: **module labels from scale-outlier
sources (gqljq 3.5%, final_mobilenet 9.5%, battery-modules 11%) were dropped** —
keeping only sources whose 'module' box matches the MTech test definition (~17%),
plus **all busbar data**. Clean MTech-only val/test.

Train module scale is now 20.6% (was dragged to a mix incl. 3.5% before).
This tests whether *consistent* multi-source data beats the 0.818 baseline.

**Setup:** Runtime -> Change runtime type -> GPU. Upload `ev_curated_data.zip`
(in your ~/Downloads) to your Google Drive (anywhere — the notebook finds it).

In [2]:
# 1. GPU + install
import torch, subprocess
print('CUDA:', torch.cuda.is_available(), '|', subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'))
!pip install -q ultralytics

CUDA: True | Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.2 MB/s eta 0:00:00


In [3]:
# 2. Mount Drive and AUTO-FIND the zip (searches common locations)
from google.colab import drive
import glob, subprocess
drive.mount('/content/drive')

hits = glob.glob('/content/drive/MyDrive/**/ev_curated_data.zip', recursive=True)
assert hits, 'ev_curated_data.zip not found in your Drive — upload it to MyDrive first.'
ZIP = hits[0]
print('Found:', ZIP)
subprocess.run(['unzip', '-q', '-o', ZIP, '-d', '/content/'])
!ls /content/ev_curated
!echo 'train images:' $(ls /content/ev_curated/images/train/*.jpg | wc -l)

Mounted at /content/drive
Found: /content/drive/MyDrive/Colab Notebooks/ev_curated_data.zip
dataset.yaml  images  labels
train images: 4425


In [4]:
# 3. Train YOLOv8n (paper-style safe augmentation; higher patience for full convergence)
from ultralytics import YOLO
YAML = '/content/ev_curated/dataset.yaml'
model = YOLO('yolov8n.pt')
model.train(
    data=YAML, epochs=150, imgsz=640, batch=32, device=0,
    optimizer='SGD', lr0=0.01, weight_decay=0.0005,
    hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5,
    degrees=2.0, translate=0.06, fliplr=0.5,
    patience=40, project='/content/runs', name='curated', exist_ok=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ev_curated/dataset.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

KeyboardInterrupt: 

In [ ]:
# 4. Evaluate on the clean MTech test set (apples-to-apples vs 0.818 baseline)
m = model.val(data=YAML, split='test', imgsz=640, device=0,
              project='/content/runs', name='curated_test', exist_ok=True)
print(f'TEST mAP50    = {m.box.map50:.3f}   (baseline 0.818)')
print(f'TEST mAP50-95 = {m.box.map:.3f}   (baseline 0.557)')
for i, c in m.names.items():
    print(f'  {c:8s} mAP50={m.box.ap50[i]:.3f}')

In [ ]:
# 5. Save weights to Drive + download
from google.colab import files
best = '/content/runs/curated/weights/best.pt'
!cp "$best" /content/drive/MyDrive/ev_detector_curated_best.pt
print('Saved to Drive: MyDrive/ev_detector_curated_best.pt')
files.download(best)

## Read the result
- If **module mAP50** is now healthy (was 0.09 on the naive merge) and overall
  test mAP50 approaches/beats 0.818, the curation worked.
- Send me the Cell 4 numbers and I'll compare + run the cross-variant probe with
  the downloaded `best.pt`.